# 🏚️ 03 — Building Damage Assessment
### Gaza Building Damage Assessment Pipeline

Runs **DOFA** on pre and post-event chips, then compares segmentation maps
to classify each area as:

| Class | Meaning |
|---|---|
| 0 — No Damage | Structure intact, same class before and after |
| 1 — Minor Damage | Class changed to adjacent type |
| 2 — Major Damage | Significant class change, likely partial collapse |
| 3 — Destroyed | Changed to bare soil / rubble |

---
**Pipeline:**
1. ✅ `01_download_data.ipynb`
2. ✅ `02_prepare_scenes.ipynb`
3. 🏚️ `03_assess_damage.ipynb` ← *you are here*
4. 🎨 `04_visualise.ipynb`


## ⚙️ Configuration

In [1]:
import sys
import numpy as np
import torch
import rasterio
import pandas as pd
import json
from pathlib import Path
from PIL import Image
import torchvision.transforms as T
import warnings
warnings.filterwarnings("ignore")

# ── Add geo-deep-learning to path ────────────────────────────────────────────
sys.path.insert(0, r"C:\geo-deep-learning")

# ── Paths ─────────────────────────────────────────────────────────────────────
CHECKPOINT_PATH = Path(r"C:\geo-deep-learning\logs\gdl_experiment\version_11\checkpoints\model-epoch=00-val_loss=0.141.ckpt")
PREPARED_DIR    = Path("data/prepared")
CHIPS_DIR       = Path("data/chips")
OUTPUT_DIR      = Path("outputs/damage")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Model config ──────────────────────────────────────────────────────────────
PATCH_SIZE         = 64
WAVELENGTHS        = torch.tensor([0.665, 0.549, 0.481])
MEAN               = [0.485, 0.456, 0.406]
STD                = [0.229, 0.224, 0.225]
PIXEL_SIZE_M       = 0.5    # Maxar 50cm resolution
HECTARES_PER_PIXEL = (PIXEL_SIZE_M ** 2) / 10_000

CLASS_NAMES = [
    "Annual Crop", "Forest", "Herbaceous Veg", "Highway",
    "Industrial", "Pasture", "Permanent Crop", "Residential",
    "River", "Sea / Lake",
]
NUM_CLASSES = len(CLASS_NAMES)

# ── Damage class definitions ──────────────────────────────────────────────────
# Maps (pre_class, post_class) pairs to damage severity
DAMAGE_CLASSES  = {0: "No Damage", 1: "Minor Damage", 2: "Major Damage", 3: "Destroyed"}
DAMAGE_COLORS   = {0: "#2ecc71", 1: "#f39c12", 2: "#e67e22", 3: "#c0392b"}

print("✅ Configuration set")
print(f"   Checkpoint exists : {CHECKPOINT_PATH.exists()}")
print(f"   Chips dir exists  : {CHIPS_DIR.exists()}")


✅ Configuration set
   Checkpoint exists : True
   Chips dir exists  : True


## 📦 Step 1 — Load DOFA Model

In [2]:
import segmentation_models_pytorch as smp
from geo_deep_learning.tasks_with_models.segmentation_dofa import SegmentationDOFA

print(f"Loading checkpoint...")
ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
hp   = ckpt["hyper_parameters"]

model = SegmentationDOFA(
    encoder       = hp["encoder"],
    pretrained    = False,
    image_size    = tuple(hp["image_size"]),
    num_classes   = hp["num_classes"],
    max_samples   = hp.get("max_samples", 2),
    loss          = smp.losses.DiceLoss(mode="multiclass", from_logits=True),
    class_labels  = hp.get("class_labels"),
    class_colors  = hp.get("class_colors"),
    freeze_layers = hp.get("freeze_layers"),
)
model.configure_model()
state = {k.removeprefix("model."): v for k, v in ckpt["state_dict"].items()}
model.model.load_state_dict(state, strict=True)
model.eval()

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ Model loaded — {total/1e6:.1f}M params ({trainable/1e6:.1f}M trainable)")


Loading checkpoint...
✅ Model loaded — 140.3M params (35.0M trainable)


## 🔍 Step 2 — Run DOFA Inference on All Chips

In [3]:
tf = T.Compose([T.Resize((PATCH_SIZE, PATCH_SIZE)), T.Normalize(mean=MEAN, std=STD)])

def infer_chip(chip_array):
    """Run DOFA on a single (3, H, W) float32 chip. Returns class ID (int)."""
    img_uint8 = (chip_array.transpose(1,2,0)*255).clip(0,255).astype(np.uint8)
    pil_img   = Image.fromarray(img_uint8)
    tensor    = tf(T.ToTensor()(pil_img)).unsqueeze(0)  # (1,3,64,64)
    wv        = WAVELENGTHS.unsqueeze(0)                 # (1,3)
    with torch.no_grad():
        output = model(tensor, wv)
        probs  = output.out.softmax(dim=1).squeeze(0)   # (C,64,64)
        cls    = probs.mean(dim=(1,2)).argmax().item()  # dominant class
    return cls

# Load chip metadata
with open(PREPARED_DIR / "chips_meta.json") as f:
    meta = json.load(f)

pre_meta  = meta["pre"]
post_meta = meta["post"]

print(f"Running inference on {len(pre_meta)} pre-event chips...")
pre_classes = []
for i, chip_info in enumerate(pre_meta):
    chip = np.load(chip_info["path"])
    pre_classes.append(infer_chip(chip))
    if (i+1) % 50 == 0:
        print(f"  {i+1}/{len(pre_meta)}", end="\r")
print(f"✅ Pre-event done  — {len(pre_classes)} chips classified")

print(f"\nRunning inference on {len(post_meta)} post-event chips...")
post_classes = []
for i, chip_info in enumerate(post_meta):
    chip = np.load(chip_info["path"])
    post_classes.append(infer_chip(chip))
    if (i+1) % 50 == 0:
        print(f"  {i+1}/{len(post_meta)}", end="\r")
print(f"✅ Post-event done — {len(post_classes)} chips classified")


Running inference on 441 pre-event chips...
✅ Pre-event done  — 441 chips classified

Running inference on 441 post-event chips...
✅ Post-event done — 441 chips classified


## 🏚️ Step 3 — Classify Damage Severity

In [ ]:
# Damage scoring logic:
# 0 = No Damage    — same class before and after
# 1 = Minor Damage — changed to a similar class (e.g. Residential → Industrial)
# 2 = Major Damage — changed to vegetation or highway (partial collapse, access issues)
# 3 = Destroyed    — changed to Annual Crop or bare land (rubble/cleared)

# Define class groups for damage logic
URBAN_CLASSES   = {4, 7}   # Industrial, Residential
VEG_CLASSES     = {0, 1, 2, 5, 6}  # Crops, Forest, Herbaceous, Pasture, Perm Crop
INFRA_CLASSES   = {3, 8, 9}  # Highway, River, Sea

def classify_damage(pre_cls, post_cls):
    if pre_cls == post_cls:
        return 0   # No damage
    if pre_cls in URBAN_CLASSES:
        if post_cls in URBAN_CLASSES:
            return 1   # Minor — still urban, slight change
        elif post_cls in INFRA_CLASSES:
            return 2   # Major — structure changed significantly
        elif post_cls in VEG_CLASSES:
            return 3   # Destroyed — urban became vegetated/bare
    return 1   # Default: minor change

damage_classes = [classify_damage(p, q) for p, q in zip(pre_classes, post_classes)]

# Summary
from collections import Counter
counts = Counter(damage_classes)
total  = len(damage_classes)
print("📊 Damage Assessment Summary")
print("-" * 40)
for cls_id, label in DAMAGE_CLASSES.items():
    n   = counts.get(cls_id, 0)
    pct = n / total * 100
    bar = "█" * int(pct / 2)
    print(f"  {label:15s}: {n:4d} chips  ({pct:5.1f}%)  {bar}")
print(f"\n  Total chips assessed: {total}")
print(f"  Chips with any damage: {sum(1 for d in damage_classes if d > 0)} ({sum(1 for d in damage_classes if d > 0)/total*100:.1f}%)")


## 🗺️ Step 4 — Reconstruct Full-Scene Damage Map

In [ ]:
# Reconstruct the damage map by placing each chip's damage class
# back into its original spatial position

scene_shape = meta["scene_shape"]   # (3, H, W)
_, H, W     = scene_shape
stride      = meta["stride"]
patch_size  = meta["patch_size"]

# Accumulate damage votes across overlapping chips
damage_votes = np.zeros((H, W, 4), dtype=np.float32)   # 4 damage classes
count_map    = np.zeros((H, W),    dtype=np.float32)

for chip_info, dmg in zip(pre_meta, damage_classes):
    y, x = chip_info["y"], chip_info["x"]
    h, w = chip_info["h"], chip_info["w"]
    damage_votes[y:y+h, x:x+w, dmg] += 1.0
    count_map[y:y+h, x:x+w]         += 1.0

# Avoid divide by zero
count_map = np.maximum(count_map, 1.0)
damage_votes /= count_map[:, :, np.newaxis]

# Final damage map: class with most votes per pixel
damage_map = damage_votes.argmax(axis=2).astype(np.uint8)

print(f"✅ Damage map reconstructed — shape: {damage_map.shape}")

# Also reconstruct pre/post segmentation maps
pre_seg_map  = np.zeros((H, W), dtype=np.int32)
post_seg_map = np.zeros((H, W), dtype=np.int32)

for chip_info, pre_cls, post_cls in zip(pre_meta, pre_classes, post_classes):
    y, x = chip_info["y"], chip_info["x"]
    h, w = chip_info["h"], chip_info["w"]
    pre_seg_map[y:y+h,  x:x+w] = pre_cls
    post_seg_map[y:y+h, x:x+w] = post_cls

print(f"   Pre  segmentation map: unique classes = {np.unique(pre_seg_map).tolist()}")
print(f"   Post segmentation map: unique classes = {np.unique(post_seg_map).tolist()}")
print(f"   Damage map           : unique classes = {np.unique(damage_map).tolist()}")


## 💾 Step 5 — Save Outputs

In [ ]:
with rasterio.open(PREPARED_DIR / "pre_rgb.tif") as src:
    profile = src.profile

def save_raster(array, path, dtype="uint8"):
    if array.ndim == 2:
        array = array[np.newaxis, :]
    p = profile.copy()
    p.update(count=array.shape[0], dtype=dtype,
             height=array.shape[1], width=array.shape[2],
             driver="GTiff", compress="lzw")
    with rasterio.open(path, "w", **p) as dst:
        dst.write(array)
    print(f"   Saved: {path}")

save_raster(damage_map,   OUTPUT_DIR / "damage_map.tif")
save_raster(pre_seg_map,  OUTPUT_DIR / "pre_segmentation.tif",  dtype="int32")
save_raster(post_seg_map, OUTPUT_DIR / "post_segmentation.tif", dtype="int32")

# Save chip-level results as CSV
results_df = pd.DataFrame([
    {
        "chip_id":      i,
        "y":            pre_meta[i]["y"],
        "x":            pre_meta[i]["x"],
        "pre_class":    CLASS_NAMES[pre_classes[i]],
        "post_class":   CLASS_NAMES[post_classes[i]],
        "damage_class": damage_classes[i],
        "damage_label": DAMAGE_CLASSES[damage_classes[i]],
    }
    for i in range(len(pre_meta))
])
csv_path = OUTPUT_DIR / "damage_results.csv"
results_df.to_csv(csv_path, index=False)
print(f"   Saved: {csv_path}")
print(f"\n✅ All outputs saved. Proceed to: 04_visualise.ipynb")
